# Heart Disease Dataset: Import and Preprocessing

This notebook loads `heart.csv` and performs preprocessing:
- data loading and quick checks
- missing-value and duplicate handling
- feature/target split
- train/test split with stratification
- one-hot encoding for categorical features
- scaling numeric features

In [4]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load dataset
file_path = "heart.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())

# Basic data quality checks
print("\nMissing values per column:")
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows before cleaning: {duplicate_count}")

Dataset shape: (1025, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0



Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Duplicate rows before cleaning: 723


In [5]:
# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

# Separate features and target
target_col = "target"
X = df.drop(columns=[target_col])
y = df[target_col]

# Define feature groups
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
numerical_features = [col for col in X.columns if col not in categorical_features]

# Build preprocessing pipelines
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit on train and transform train/test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Create readable processed DataFrames
feature_names = preprocessor.get_feature_names_out()
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)

print("\nProcessed train shape:", X_train_processed_df.shape)
print("Processed test shape:", X_test_processed_df.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True))

# Optional: save processed datasets
train_output = pd.concat([X_train_processed_df.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
test_output = pd.concat([X_test_processed_df.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

train_output.to_csv("heart_train_preprocessed.csv", index=False)
test_output.to_csv("heart_test_preprocessed.csv", index=False)

print("\nSaved files:")
print("- heart_train_preprocessed.csv")
print("- heart_test_preprocessed.csv")

display(train_output.head())

Shape after removing duplicates: (302, 14)

Processed train shape: (241, 30)
Processed test shape: (61, 30)
Train target distribution:
target
1    0.543568
0    0.456432
Name: proportion, dtype: float64

Saved files:
- heart_train_preprocessed.csv
- heart_test_preprocessed.csv


,num__age,num__trestbps,num__chol,num__thalach,num__oldpeak,cat__sex_0,cat__sex_1,cat__cp_0,cat__cp_1,cat__cp_2,...,cat__ca_0,cat__ca_1,cat__ca_2,cat__ca_3,cat__ca_4,cat__thal_0,cat__thal_1,cat__thal_2,cat__thal_3,target
0,1.421944,-0.973041,5.882908,0.408240,0.527263,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,-0.465841,0.756507,-0.885696,-1.104705,-0.083233,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
2,0.422528,-0.197726,-0.588175,-0.882213,1.050546,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
3,1.644036,0.517949,0.118437,-0.214737,0.876119,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
4,-1.021072,-1.151959,-1.629499,-0.570724,-0.868157,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
